In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import time

d:\codes\summarizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

t1 = time.time()
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
print(f"Model loaded in {time.time() - t1:.2f} seconds.")

Loading model...


d:\codes\summarizer\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shahn\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For

Model loaded in 97.55 seconds.


In [ ]:
prompt = "question: What is product of 2 and 9 ? Answer:"


In [19]:
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

a product of two


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 2. Device Setup (Crucial for performance)
# Check if a GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Load Tokenizer and Model
# Load the tokenizer and the Seq2Seq model, then move the model to the determined device
tokenizer_s = AutoTokenizer.from_pretrained(MODEL_ID)
model_s = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)
print(f"Model {MODEL_ID} loaded successfully.")
# 1. Configuration
MODEL_ID = "google/flan-t5-small"



Using device: cpu
Model google/flan-t5-small loaded successfully.


In [63]:
article_to_summarize = """
The James Webb Space Telescope (JWST) has captured a stunning new image of the Pillars of Creation. 
The famous pillars, located in the Eagle Nebula, about 6,500 light-years from Earth, were first immortalized by 
the Hubble Space Telescope in 1995. Webb’s near-infrared camera sees through much of the dust that obscured 
the stars in the original Hubble image, revealing thousands of newly formed protostars. These young stars 
are still gathering mass, and their formation process involves shooting out powerful supersonic jets of material 
that interact with the pillars, causing the luminous, wavy patterns seen at the edges. Astronomers use these 
observations to better understand how stars form and the conditions of their stellar nurseries.
"""

text = """
Which category does this belong to? options: [Sports, Politics, Finance]. Text: The stock market closed 2% higher today
"""

In [64]:
T5_PREFIX = "Find keywords : " # T5 requires a prefix to know the task
MAX_OUTPUT_LENGTH = 500


# 5. Tokenize the Input Manually
# A. Add the prefix to the article
input_text = text

# B. Tokenize the text
# return_tensors='pt' ensures the output is a PyTorch tensor
# truncation=True is essential for long articles
input_ids = tokenizer(
    input_text, 
    return_tensors="pt", 
    max_length=512, 
    truncation=True
).input_ids.to(device) # C. Move the input tensor to the same device as the model

# 6. Generate the Summary (Inference)
# This is where the magic happens—the model generates the output sequence
outputs = model.generate(
    input_ids,
    max_length=MAX_OUTPUT_LENGTH,      # Max length of the summary
    num_beams=4,                       # Use Beam Search for higher quality output
    early_stopping=True                # Stop when all beam hypotheses are complete
)
print(outputs[0])

# 7. Decode the Output
# Convert the output token IDs back into a human-readable string
base_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- Manual Flan-T5 Small Summary (Using Beam Search) ---")
print(base_summary)
print("----------------------------------------------------------")

tensor([   0, 7679,    1])

--- Manual Flan-T5 Small Summary (Using Beam Search) ---
Finance
----------------------------------------------------------


Result from running the t5-base-span model i find that it gives same output when input is not changing. What it means is that model is not predicting each word evertime. but has a set of weights for a specific combination so it gives that answer.